# 02 · Delta Lake — aplicar las devoluciones con `MERGE INTO`

**Pregunta que se contesta aquí:** ¿cuáles de las líneas que ya están en Silver terminaron siendo devueltas?

## Qué pasa en este notebook

En Silver todas las líneas tienen `fue_devuelto = 0`, porque al momento de la compra nadie sabe si la prenda volverá. Días después llega un lote de devoluciones. Este notebook lo aplica y deja evidencia de cuatro operaciones de Delta Lake:

1. **`MERGE INTO`**: actualiza solo las líneas afectadas por el lote de devoluciones.
2. **`DESCRIBE HISTORY`**: muestra que el `MERGE` quedó registrado como una versión nueva de la tabla.
3. **Time Travel (`VERSION AS OF`)**: permite ver la tabla *antes* y *después* del `MERGE` y comparar.
4. **`OPTIMIZE ... ZORDER BY`**: compacta los archivos que el `MERGE` dejó fragmentados.

Estas operaciones funcionan porque las tablas de Silver son Delta: archivos Parquet más un registro de transacciones (`_delta_log`) que da garantías ACID. Un `MERGE` que falla a mitad de camino no deja la tabla a medias.

**En esta celda** se hereda la configuración compartida.

In [0]:
%run ./00_Setup/00_Configuracion

**En esta celda** se importan las funciones de PySpark.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql.functions import broadcast

**En esta celda** se deja el estado inicial de `silver.lineas_pedido`: cuántas líneas hay, cuántas figuran como devueltas (ninguna todavía), cuántos archivos tiene y su historial de versiones. Es el "antes" contra el que se compara después del `MERGE`.

In [0]:
TABLA = tbl(SCHEMA_SILVER, "lineas_pedido")

det_antes = spark.sql(f"DESCRIBE DETAIL {TABLA}").first()
n_lineas = spark.table(TABLA).count()
n_devueltas_antes = spark.table(TABLA).filter("fue_devuelto = 1").count()

print(f"Tabla: {TABLA}")
print(f"  lineas                : {n_lineas:,}")
print(f"  marcadas como devueltas: {n_devueltas_antes:,}")
print(f"  numFiles              : {det_antes['numFiles']}")
display(spark.sql(f"DESCRIBE HISTORY {TABLA}").select("version", "timestamp", "operation"))

**En esta celda** se prepara el lote de devoluciones: se lee `bronze.devoluciones`, se asegura que haya una sola fila por `linea_id` y se registra como vista temporal para poder usarla dentro del `MERGE`.

**Por qué la deduplicación:** un `MERGE` falla si dos filas del origen coinciden con la misma fila del destino. Garantizar una fila por `linea_id` evita ese error.

In [0]:
df_lote_devoluciones = (
    spark.table(tbl(SCHEMA_BRONZE, "devoluciones"))
    .dropDuplicates(["linea_id"])
    .select("linea_id", "fecha_devolucion")
)
df_lote_devoluciones.createOrReplaceTempView("lote_devoluciones")
print(f"Lote de devoluciones listo: {df_lote_devoluciones.count():,} registros")

**En esta celda** se aplica el `MERGE INTO`. Cuando una línea del lote coincide con una línea de Silver (`ON destino.linea_id = origen.linea_id`), se guarda la fecha de devolución y `fue_devuelto` pasa a 1 solo si la devolución llegó dentro de la ventana de 30 días (`datediff(devolucion, compra) <= 30`). Las que llegaron más tarde conservan la fecha, pero la etiqueta queda en 0.

**Por qué `MERGE` y no sobrescribir la tabla:** el lote trae devoluciones para una fracción de las líneas. Sobrescribir obligaría a reescribir todas las filas cada vez que llega un lote. El `MERGE` toca solo los archivos afectados, deja el cambio registrado como una versión nueva (lo que habilita el Time Travel) y es **idempotente**: correrlo dos veces deja la tabla igual, así que un reintento automático de un Job no duplica nada.

In [0]:
spark.sql(f"""
    MERGE INTO {TABLA} AS destino
    USING lote_devoluciones AS origen
    ON destino.linea_id = origen.linea_id
    WHEN MATCHED THEN UPDATE SET
        fecha_devolucion = origen.fecha_devolucion,
        fue_devuelto = CASE WHEN datediff(origen.fecha_devolucion, destino.fecha) <= {VENTANA_DEVOLUCION_DIAS} THEN 1 ELSE 0 END
""")

print("MERGE aplicado. Metricas de la operacion:")
metricas = spark.sql(f"DESCRIBE HISTORY {TABLA}").filter("operation = 'MERGE'").first()["operationMetrics"]
for k in ["numTargetRowsUpdated", "numTargetRowsMatchedUpdated", "numTargetRowsInserted", "numOutputRows"]:
    if k in metricas:
        print(f"  {k:<28} {metricas[k]}")

n_devueltas_despues = spark.table(TABLA).filter("fue_devuelto = 1").count()
print(f"\nLineas con fue_devuelto = 1: {n_devueltas_despues:,} de {n_lineas:,} ({n_devueltas_despues / n_lineas:.1%})")

**En esta celda** se usa Time Travel para ver la tabla antes y después del `MERGE`. Se busca en el historial la versión del `MERGE` y la anterior, y se cuentan las líneas devueltas en cada una con `VERSION AS OF`. Así se demuestra que el cambio quedó registrado y que el estado previo sigue disponible sin necesidad de copias de respaldo.

In [0]:
hist = spark.sql(f"DESCRIBE HISTORY {TABLA}").collect()
v_merge = [h["version"] for h in hist if h["operation"] == "MERGE"][0]   # ultima version creada por un MERGE
v_antes = v_merge - 1

devueltas_antes  = spark.sql(f"SELECT COUNT(*) FROM {TABLA} VERSION AS OF {v_antes} WHERE fue_devuelto = 1").first()[0]
devueltas_despues = spark.sql(f"SELECT COUNT(*) FROM {TABLA} VERSION AS OF {v_merge} WHERE fue_devuelto = 1").first()[0]

print(f"{'':36} {'Version ' + str(v_antes):>12} {'Version ' + str(v_merge):>12}")
print(f"{'Lineas con fue_devuelto = 1':36} {devueltas_antes:>12,} {devueltas_despues:>12,}")
print(f"\nLineas reclasificadas por el MERGE: {devueltas_despues - devueltas_antes:,}")
print("\nPara deshacer el MERGE completo (rollback) bastaria:")
print(f"  RESTORE TABLE {TABLA} TO VERSION AS OF {v_antes}")
print("  (no se ejecuta aqui: las demas capas dependen del estado actual)")

**En esta celda** se compacta la tabla con `OPTIMIZE ... ZORDER BY (cliente_id)` y se compara el número de archivos antes y después.

**Por qué `OPTIMIZE`:** cada `MERGE` reescribe los archivos que toca y deja archivos pequeños; muchos archivos pequeños hacen lenta la lectura. `OPTIMIZE` los compacta en pocos archivos de buen tamaño.

**Por qué `ZORDER BY (cliente_id)`:** en Gold las features se calculan agrupando y uniendo por `cliente_id`. Es una columna de alta cardinalidad (miles de valores), así que no se puede usar para particionar sin crear miles de carpetas diminutas. `ZORDER` reorganiza los datos dentro de los archivos para que las filas de un mismo cliente queden cerca, y Spark puede saltarse archivos al filtrar por esa columna. Con solo 20 mil filas la mejora es pequeña; lo que importa aquí es el patrón, que se vuelve decisivo con volúmenes reales.

In [0]:
det_pre = spark.sql(f"DESCRIBE DETAIL {TABLA}").first()
print(f"Antes de OPTIMIZE: {det_pre['numFiles']} archivos, tamano medio {det_pre['sizeInBytes'] / max(det_pre['numFiles'], 1) / 1024:.1f} KB")

resultado = spark.sql(f"OPTIMIZE {TABLA} ZORDER BY (cliente_id)")
display(resultado)

det_post = spark.sql(f"DESCRIBE DETAIL {TABLA}").first()
print(f"\nnumFiles antes -> despues: {det_pre['numFiles']} -> {det_post['numFiles']}")
display(spark.sql(f"DESCRIBE HISTORY {TABLA}").select("version", "timestamp", "operation"))

Delta Lake · `MERGE`, Time Travel y `OPTIMIZE`